# Overview

We are going to use ASE to calculate the thermochemistry of an ideal gas. ASE also contains various routines to calculate the thermochemistry of solids and molecular adsorbates on surfaces, but we will focus on lone molecules for the time being.

For this exercise, we will use a brand new machine learning potential called [UPET](https://github.com/lab-cosmo/upet). This is a machine learning surrogate for DFT, which we will learn about later in the course.


# Setup


In [ ]:
!uv pip install upet

Until a [bug is fixed](https://github.com/lab-cosmo/upet/issues/99), you will need to download the ML model checkpoint yourself from [HuggingFace](https://huggingface.co/lab-cosmo/upet/tree/main/models). Please download `pet-mad-s-v1.5.0.ckpt` and put it in the current working directory. I also tried to automate this below, but there are no guarantees.


In [ ]:
!curl -L "https://huggingface.co/lab-cosmo/upet/resolve/main/models/pet-mad-s-v1.5.0.ckpt" -o pet-mad-s-v1.5.0.ckpt

# Demonstration


First, we are going to instantiate the ASE calculator. As a reminder, this tells ASE how the energies and forces will be calculated. It could be any method, such as DFT, but here we will use the faster UPET machine learned interatomic potential.


In [ ]:
from upet.calculator import UPETCalculator

calc = UPETCalculator(checkpoint_path="pet-mad-s-v1.5.0.ckpt")

Now we are going to pick a system to study. Let's go through this exercise for N2O. Our goal is to go from E (i.e. 0 K electronic energy) to G (i.e. Gibbs free energy).


First, we need to make sure we optimize our structure so that it is a minimum in the potential energy landscape. We do that by first defining our molecule, attaching the calculator to it, and running the optimization. We do not need to wrap the molecule in a `FrechetCellFilter` here because there is no unit cell to optimize.


In [ ]:
from ase.build import molecule
from ase.optimize import BFGS

atoms = molecule("N2O")
atoms.calc = calc
opt = BFGS(atoms)
opt.run(fmax=1)

Great! We have optimized the geometry. Now, we are going to make sure we fetch the final energy (E), which we will need for later.


In [ ]:
E = atoms.get_potential_energy()
print(E)

In [ ]:
from ase.visualize import view

view(atoms, viewer="x3d")

To go from an energy to Gibbs free energy, we need the vibrational modes. For that, we must run a vibrational frequency analysis. This has the added benefit of telling us if we are at a minimum in the potential energy landscape as intended.


In [ ]:
from ase.vibrations import Vibrations

vib = Vibrations(atoms)
vib.clean()
vib.run()

Note that we should expect there to be 3N-5 vibrational modes for this molecule since it is linear (if it were nonlinear, there would be 3N-6). ASE does not remove any of these spurious modes in the vibrational summary printout.


In [ ]:
print(f"There should be {3 * len(atoms) - 5} real vibrational modes.")

In [ ]:
vib.summary()

We can already see that the zero-point vibrational energy (E_ZPVE) was computed based on the vibrational frequencies. That's convenient, but we have to be careful because not all of thsoe modes are physically relevant.


Now we will go ahead and correct E to G(T, P) using the vibrational analysis. ASE comes with many [thermochemistry](https://ase-lib.org/ase/thermochemistry/thermochemistry.html) modules. You should check out the documentation. It includes all the equations and also is generally useful.

We are going to treat the molecule as an ideal gas and so we will use the `IdealGasThermo` class.


In [ ]:
from ase.thermochemistry import IdealGasThermo

We have to pass several arguments to the `IdealGasThermoClass`. The vibrational energies are needed for the vibrational heat capacity. We also need to specify whether it is linear or not to get the correct rotational heat capacity. The `potentialenergy` is the E value that the correction will be added to. The `symmetrynumber` is the number of symmetric rotations that the molecule has (it is often easiest to just look this up, although there are symmetry tools to automate this). The `spin` is the 1/2 \* the number of unpaired electrons, which is 0 here.

Note that the spurious vibrational modes will be "cut" automatically by the `IdealGasThermo` class, so you do not need to worry about that.


In [ ]:
vib_energies = vib.get_energies()
igt = IdealGasThermo(
    vib_energies, "linear", potentialenergy=E, atoms=atoms, symmetrynumber=1, spin=0
)

Now we can get various properties. Let's start with the E_ZPVE.


In [ ]:
igt.get_ZPE_correction()

Now let's get the enthalpy and entropy. We will need to specify the conditions.


In [ ]:
T = 298.15  # K
P = 1e5  # Pa

In [ ]:
H = igt.get_enthalpy(T)

In [ ]:
S = igt.get_entropy(T, P)

If we'd like, we can calculate G manually.


In [ ]:
G_manual = H - T * S
print(G_manual)

We can also just request it directly.


In [ ]:
G = igt.get_gibbs_energy(T, P)
print(G)

Finally, we can conclude by noting what the magnitude of this Gibbs free energy correction was.


In [ ]:
print(f"The difference between G and E is {G - E} eV")

This is a fairly large difference. The magnitude of the difference will typically be smaller for solids since they lack translational and rotational entropy contributions, but it can still be important.


Now let's see what happens when we bump up the temperature.


In [ ]:
G_1000 = igt.get_gibbs_energy(1000, P, verbose=False)
print(f"The difference between G(1000 K) and G(298.15 K) is {G_1000 - G} eV")